# Nexus sleeve attribution (v2)

Running inventory (BOD `algo.position` + transfers/fills) with no-roll sleeves:

`overnight + timing_mr + pca_held_mr + fill_mr + mark_to_fill + unexecuted_mr`

Outbound Hedger → Spreadgr Base 2 is excluded from the benchmark.


In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path('/home/ajain/apeksha/pnl_attribution/src/v2')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import CONFIG, get_ch_client
from scripts.library import load_library_eod_pnl
from scripts.position_timeline import load_bod_spreads, reconcile_timeline_vs_library
from scripts.sleeve_attribution import (
    SLEEVE_COLS,
    assign_transfers_to_cycles,
    attribute_day_sleeves,
    attribute_range_vs_library,
    compute_timing_mr,
)

client = get_ch_client()
BOOK = 'Hedger Spreads'
DAY = date(2026, 10, 5)
START = date(2026, 9, 17)

print('books =', len(CONFIG.book_names))
print('book =', BOOK, '| day =', DAY, '| range from', START)


### Inventory + library recon

BOD spreads and event-walk gross vs library (`overnight + m2m + trade`).


In [ ]:
bod_ts, bod = load_bod_spreads(client, DAY, BOOK)
print('BOD curve', bod_ts)
display(bod.to_frame('lots').T)

recon = reconcile_timeline_vs_library(client, DAY, BOOK)
display(recon)
lib = load_library_eod_pnl(client, DAY, BOOK)
print('library', lib)


### Assign transfers + timing MR

- Keep recorded `cycle_id`
- Untagged → first intention after the transfer
- Never assign Hedger → Spreadgr Base 2 rollers
- `timing_mr = lots × 1000 × (curve_cycle − curve_xfer)`


In [ ]:
assigned = assign_transfers_to_cycles(client, BOOK, START, DAY)
assigned = compute_timing_mr(client, assigned)
print(assigned['assigned_from'].value_counts(dropna=False).to_dict())
print('timing_mr sum', round(float(assigned['timing_mr'].sum()), 2))
display(assigned.head(10))


### Day sleeves on the running book

Stamps overnight (once), timing, PCA held, fill MR, mark-to-fill, unexecuted MR.


In [ ]:
tl_mr, totals = attribute_day_sleeves(client, DAY, BOOK, assigned)
lib_g = round(float(lib['gross']), 2) if lib else float('nan')
print({k: round(totals[k], 2) for k in (*SLEEVE_COLS, 'ours')})
print('library_gross', lib_g, '| diff', round(totals['ours'] - lib_g, 2))

cols = [
    'timestamp', 'reason', 'cycle_id', 'source_book', 'outbound',
    'overnight', 'timing_mr', 'pca_held_mr',
    'fill_mr', 'mark_to_fill', 'unexecuted_mr',
]
display(tl_mr[[c for c in cols if c in tl_mr.columns]].head(20))


### Range check vs library (from 17 Sep)

No-roll benchmark. Re-run after tomorrow's EOD lands.


In [ ]:
cmp = attribute_range_vs_library(client, BOOK, START, end=DAY)
display(cmp)
if not cmp.empty:
    print('days', len(cmp),
          '| mean |diff|', round(cmp['diff'].abs().mean(), 2),
          '| max |diff|', round(cmp['diff'].abs().max(), 2))
